In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print(torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print(device)

In [ ]:
df = pd.read_csv("clusters_7x7.csv")
results_df = pd.read_csv("cluster_features.csv")

print(df.shape)
print(results_df.shape)

df.head()

对比ADC的Range0~1000，和对ADC的值取对数，看数据分布
数据分布越合理
训练效果越好

In [ ]:
adc_test = np.array([
    0,
    1,
    5,
    10,
    20,
    50,
    100,
    200,
    500,
    1000
])

adc_log = np.log1p(adc_test)

fig, axes = plt.subplots(
    1, 2,
    figsize=(12,5)
)

# 左图：原始ADC
axes[0].bar(
    range(len(adc_test)),
    adc_test
)

axes[0].set_xticks(
    range(len(adc_test))
)

axes[0].set_xticklabels(
    adc_test,
    rotation=45
)

axes[0].set_title(
    "Original ADC Values"
)

axes[0].set_xlabel(
    "ADC"
)

axes[0].set_ylabel(
    "Value"
)

# 右图：log(ADC+1)
axes[1].bar(
    range(len(adc_test)),
    adc_log
)

axes[1].set_xticks(
    range(len(adc_test))
)

axes[1].set_xticklabels(
    adc_test,
    rotation=45
)

axes[1].set_title(
    "log(ADC + 1)"
)

axes[1].set_xlabel(
    "Original ADC"
)

axes[1].set_ylabel(
    "Transformed Value"
)

plt.tight_layout()
plt.show()

In [ ]:
pixel_cols = [f"pixel_{i}" for i in range(49)]

X = df[pixel_cols].values.astype(np.float32)

# log(ADC+1)
X = np.log1p(X)

normal_ids = results_df[
    results_df["label"] == "Normal"
]["cluster_id"].values.astype(int)

X_train = X[normal_ids]

X_train = torch.tensor(X_train, dtype=torch.float32)

train_loader = DataLoader(
    TensorDataset(X_train),
    batch_size=32,
    shuffle=True
)

print("Total clusters:", len(X))
print("Normal training clusters:", len(X_train))

定义autoencoder

In [ ]:
class PatchAutoencoder(nn.Module):

    def __init__(self):

        super().__init__()

        self.encoder = nn.Sequential(

            nn.Linear(49, 32),
            nn.ReLU(),

            nn.Linear(32, 16),
            nn.ReLU(),

            nn.Linear(16, 2)
        )

        self.decoder = nn.Sequential(

            nn.Linear(2, 16),
            nn.ReLU(),

            nn.Linear(16, 32),
            nn.ReLU(),

            nn.Linear(32, 49)
        )

    def forward(self, x):

        z = self.encoder(x)

        x_recon = self.decoder(z)

        return x_recon

创建模型并训练

In [ ]:
model = PatchAutoencoder().to(device)

loss_fn = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

epochs = 200

loss_history = []

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for batch in train_loader:

        x = batch[0].to(device)

        x_recon = model(x)

        loss = loss_fn(
            x_recon,
            x
        )

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    loss_history.append(avg_loss)

    if (epoch + 1) % 10 == 0:

        print(
            f"Epoch {epoch+1:3d} "
            f"Loss={avg_loss:.6f}"
        )

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(
    loss_history,
    linewidth=2
)

plt.xlabel("Epoch")
plt.ylabel("Average Training Loss")
plt.title(
    "Training Loss of Autoencoder"
)
plt.grid(True)
plt.show()

编码所有的cluster

In [ ]:
X_all = torch.tensor(
    X,
    dtype=torch.float32
).to(device)

model.eval()

with torch.no_grad():

    Z = (
        model.encoder(X_all)
        .cpu()
        .numpy()
    )

    X_recon = (
        model(X_all)
        .cpu()
        .numpy()
    )

results_df["z1"] = Z[:,0]
results_df["z2"] = Z[:,1]

results_df["reconstruction_loss"] = np.mean(
    (X - X_recon)**2,
    axis=1
)

results_df.head()

画2D Latent Space

In [ ]:
plt.figure(figsize=(8,6))

plt.scatter(
    results_df["z1"],
    results_df["z2"],
    s=20
)

plt.xlabel("z1")
plt.ylabel("z2")

plt.title(
    "2D Latent Space"
)

plt.show()

In [ ]:
def show_reconstruction(cluster_id):

    x = torch.tensor(
        X[cluster_id].reshape(1, 49),
        dtype=torch.float32
    ).to(device)

    model.eval()

    with torch.no_grad():

        z = model.encoder(x)

        x_recon = model.decoder(z)

    original = (
        x.cpu()
        .numpy()
        .reshape(7, 7)
    )

    reconstructed = (
        x_recon.cpu()
        .numpy()
        .reshape(7, 7)
    )

    loss = np.mean(
        (original - reconstructed) ** 2
    )

    label = results_df.loc[
        results_df["cluster_id"] == cluster_id,
        "label"
    ].values[0]

    fig, axes = plt.subplots(
        1, 2,
        figsize=(8, 4)
    )

    im0 = axes[0].imshow(
        original,
        origin="lower"
    )

    axes[0].set_title(
        f"Original\nCluster {cluster_id}"
    )

    plt.colorbar(
        im0,
        ax=axes[0]
    )

    im1 = axes[1].imshow(
        reconstructed,
        origin="lower"
    )

    axes[1].set_title(
        f"Reconstructed\n{label}\nLoss={loss:.4f}"
    )

    plt.colorbar(
        im1,
        ax=axes[1]
    )

    plt.tight_layout()
    plt.show()

In [ ]:
normal_ids = results_df[
    results_df["label"] == "Normal"
]["cluster_id"].values[:6]

for cluster_id in normal_ids:

    show_reconstruction(
        int(cluster_id)
    )

保存对比图

In [ ]:
from pathlib import Path

save_dir = Path("reconstruction_gallery")
save_dir.mkdir(exist_ok=True)

normal_ids = results_df[
    results_df["label"] == "Normal"
]["cluster_id"].values[:6]

for cluster_id in normal_ids:

    cluster_id = int(cluster_id)

    x = torch.tensor(
        X[cluster_id].reshape(1, 49),
        dtype=torch.float32
    ).to(device)

    model.eval()

    with torch.no_grad():
        z = model.encoder(x)
        x_recon = model.decoder(z)

    original = x.cpu().numpy().reshape(7, 7)

    reconstructed = (
        x_recon.cpu()
        .numpy()
        .reshape(7, 7)
    )

    loss = np.mean(
        (original - reconstructed) ** 2
    )

    fig, axes = plt.subplots(
        1, 2,
        figsize=(8, 4)
    )

    im0 = axes[0].imshow(
        original,
        origin="lower"
    )

    axes[0].set_title(
        f"Original\nCluster {cluster_id}"
    )

    plt.colorbar(
        im0,
        ax=axes[0]
    )

    im1 = axes[1].imshow(
        reconstructed,
        origin="lower"
    )

    axes[1].set_title(
        f"Reconstructed\nLoss={loss:.4f}"
    )

    plt.colorbar(
        im1,
        ax=axes[1]
    )

    plt.tight_layout()

    filename = save_dir / f"normal_cluster_{cluster_id}_reconstruction.png"

    fig.savefig(
        filename,
        dpi=200,
        bbox_inches="tight"
    )

    plt.close(fig)

print("Saved reconstruction images to:", save_dir)